# Week 8: Multi-Agent Risk-Level Debate (Low / Medium / High)

## Deliberative multi-agent architecture for project/loan risk classification

This notebook implements a debate system where:
- **Advocate agents** (Low Risk, Medium Risk, High Risk) build evidence-based cases from project/loan descriptions
- **Judge agent** scores arguments on a 5-dimension rubric and decides the risk level
- **Gradio UI** shows the debate transcript and final verdict


In [ ]:
# Core imports
import os
import json
import random
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
import gradio as gr


load_dotenv(override=True)



logging.basicConfig(level=logging.INFO, format="%(message)s")
print("All imports successful")

In [ ]:
# Configuration
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

GPT_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7
AMBIGUITY_MARGIN = 0.08

openai_client = OpenAI()
print("OpenAI client initialized")

In [ ]:
# Path to risk cases CSV
DATA_PATH = Path("risk_cases.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("week8/community_contributions/risk_cases.csv")

df = pd.read_csv(DATA_PATH)
df["risk_level"] = df["risk_level"].str.strip().str.lower()
risk_cases = list(df.to_dict("records"))

# Train/val/test split (for evaluation)
n = len(risk_cases)
train_cases = risk_cases[: int(0.7 * n)]
val_cases = risk_cases[int(0.7 * n) : int(0.85 * n)]
test_cases = risk_cases[int(0.85 * n) :]

print(f"Loaded {len(risk_cases)} risk cases")
print(f"Train: {len(train_cases)}, Val: {len(val_cases)}, Test: {len(test_cases)}")
print(df["risk_level"].value_counts())

In [ ]:
@dataclass
class RiskInput:
    """Input for risk debate: project/loan description and optional metadata."""
    description: str
    project_name: Optional[str] = None
    sector: Optional[str] = None
    amount: Optional[float] = None

    def to_dict(self) -> Dict:
        return {k: v for k, v in asdict(self).items() if v is not None}


@dataclass
class Argument:
    """An advocate's argument for a risk level."""
    agent_name: str
    risk_level: str
    reasoning: str
    evidence: List[str]
    confidence: float
    acknowledges_weaknesses: bool


@dataclass
class JudgeScore:
    """Judge's evaluation of one argument."""
    agent_name: str
    evidence_grounding: float
    internal_consistency: float
    domain_plausibility: float
    counterpoint_handling: float
    confidence_calibration: float
    total_score: float


@dataclass
class JudgeDecision:
    """Final judge decision."""
    final_label: str
    explanation: str
    scores: List[JudgeScore]
    is_ambiguous: bool
    clarification_question: Optional[str] = None


print("Data structures defined")

In [ ]:
class Agent:
    """Base agent with colored logging."""
    RED = "\033[31m"
    GREEN = "\033[32m"
    YELLOW = "\033[33m"
    BLUE = "\033[34m"
    MAGENTA = "\033[35m"
    CYAN = "\033[36m"
    WHITE = "\033[37m"
    BG_BLACK = "\033[40m"
    RESET = "\033[0m"
    BOLD = "\033[1m"

    def __init__(self, name: str, color: str):
        self.name = name
        self.color = color

    def log(self, message: str):
        color_code = self.BG_BLACK + self.color + self.BOLD
        formatted = f"{color_code}[{self.name}]{self.RESET} {message}"
        logging.info(formatted)
        return formatted

In [ ]:
class AdvocateAgent(Agent):
    """Base advocate: builds evidence-based argument for a risk level."""

    def __init__(self, name: str, risk_level: str, color: str, client: OpenAI):
        super().__init__(name, color)
        self.risk_level = risk_level
        self.client = client

    def get_system_prompt(self) -> str:
        raise NotImplementedError

    def build_argument(self, risk_input: RiskInput) -> Argument:
        self.log(f"Building argument for {self.risk_level} risk...")
        system_prompt = self.get_system_prompt()
        user_prompt = f"""Project/loan to assess:
{json.dumps(risk_input.to_dict(), indent=2)}

Build the strongest evidence-based case that this case is **{self.risk_level.upper()}** risk. Output valid JSON only."""

        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        confidence = result.get("confidence", 50)
        if confidence > 1:
            confidence = confidence / 100.0
        argument = Argument(
            agent_name=self.name,
            risk_level=self.risk_level,
            reasoning=result["reasoning"],
            evidence=result.get("evidence", []),
            confidence=float(confidence),
            acknowledges_weaknesses=result.get("acknowledges_weaknesses", False),
        )
        self.log(f"Confidence: {argument.confidence:.0%}")
        return argument

In [ ]:
class LowRiskAdvocateAgent(AdvocateAgent):
    """Argues that the project/loan is LOW risk."""

    def __init__(self, client: OpenAI):
        super().__init__("LowRiskAdvocate", "low", Agent.GREEN, client)

    def get_system_prompt(self) -> str:
        return """You are LowRiskAdvocate. Build the strongest case that this project/loan is LOW risk.

DECISION-MAKING STRATEGY:
1. TRACK RECORD (Critical): Long tenure, proven management, completed projects → low risk.
2. COLLATERAL & SECURITY (Critical): Strong collateral, guarantees, secured assets → low risk.
3. CASH FLOW (High): Stable revenue, recurring income, low leverage → low risk.
4. SECTOR & MARKET (Medium): Mature sector, regulated, predictable demand → low risk.
5. MITIGANTS (Medium): Guarantees, insurance, government backing → low risk.

RULES: Cite only facts from the description. Acknowledge weaknesses honestly. Calibrate confidence (80–95% strong case, 60–79% good, 40–59% mixed, 20–39% weak).

OUTPUT (JSON only):
{"reasoning": "...", "evidence": ["fact1", "fact2", "fact3"], "confidence": 75, "acknowledges_weaknesses": true}"""

In [ ]:
class MediumRiskAdvocateAgent(AdvocateAgent):
    """Argues that the project/loan is MEDIUM risk."""

    def __init__(self, client: OpenAI):
        super().__init__("MediumRiskAdvocate", "medium", Agent.YELLOW, client)

    def get_system_prompt(self) -> str:
        return """You are MediumRiskAdvocate. Build the strongest case that this project/loan is MEDIUM risk.

DECISION-MAKING STRATEGY:
1. BALANCED PROFILE (Critical): Some strengths (experience, collateral) and some concerns (sector, leverage) → medium.
2. MAINSTREAM POSITIONING (High): Neither very safe nor very speculative; typical commercial risk.
3. MITIGATED WEAKNESSES (High): Risks present but partially offset by guarantees, track record, or structure.
4. SECTOR NORMS (Medium): Sector that typically carries moderate risk (e.g. expansion, franchise).
5. GOLDILOCKS: Not low-risk (no strong collateral + track record) but not high-risk (no startup, no volatile sector).

RULES: Cite only facts. Acknowledge factors pointing to low OR high. Calibrate confidence.

OUTPUT (JSON only):
{"reasoning": "...", "evidence": ["fact1", "fact2", "fact3"], "confidence": 70, "acknowledges_weaknesses": true}"""

In [ ]:
class HighRiskAdvocateAgent(AdvocateAgent):
    """Argues that the project/loan is HIGH risk."""

    def __init__(self, client: OpenAI):
        super().__init__("HighRiskAdvocate", "high", Agent.MAGENTA, client)

    def get_system_prompt(self) -> str:
        return """You are HighRiskAdvocate. Build the strongest case that this project/loan is HIGH risk.

DECISION-MAKING STRATEGY:
1. WEAK COLLATERAL / GUARANTEES (Critical): Unsecured, personal guarantee only, volatile collateral → high risk.
2. TRACK RECORD (Critical): New borrower, startup, unproven model, pre-revenue → high risk.
3. SECTOR & REGULATION (High): Volatile sector, regulatory uncertainty, competitive pressure → high risk.
4. LEVERAGE & CASH FLOW (High): High burn, unclear revenue, high debt → high risk.
5. EXECUTION RISK (Medium): Complex project, first-time, single point of failure → high risk.

RULES: Cite only facts. Acknowledge any strengths honestly. Calibrate confidence.

OUTPUT (JSON only):
{"reasoning": "...", "evidence": ["fact1", "fact2", "fact3"], "confidence": 80, "acknowledges_weaknesses": true}"""

In [ ]:
class JudgeAgent(Agent):
    """Impartial judge: 5-dimension rubric, winner = best argument quality."""

    def __init__(self, client: OpenAI):
        super().__init__("Judge", Agent.CYAN)
        self.client = client

    def get_judge_prompt(self) -> str:
        return f"""You are an impartial Judge evaluating risk-level arguments. Risk levels: LOW, MEDIUM, HIGH.

Score each argument on 5 dimensions (0.0–1.0):
1. EVIDENCE GROUNDING: Claims map to facts in the description (0.9–1.0 strong, 0–0.4 invented).
2. INTERNAL CONSISTENCY: No contradictions, clear logic.
3. DOMAIN PLAUSIBILITY: Reasonable risk/finance reasoning.
4. COUNTERPOINT HANDLING: Acknowledges weaknesses honestly.
5. CONFIDENCE CALIBRATION: Confidence matches evidence strength.

Total score = sum of 5 (max 5.0). Winner = HIGHEST total. If top two within {AMBIGUITY_MARGIN}, set is_ambiguous true and optionally ask ONE clarification_question. Evaluate ARGUMENT QUALITY.

OUTPUT (JSON only):
{{"scores": [{{"agent_name": "...", "evidence_grounding": 0.8, "internal_consistency": 0.8, "domain_plausibility": 0.8, "counterpoint_handling": 0.8, "confidence_calibration": 0.8, "total_score": 4.0}}, ...], "final_label": "low"|"medium"|"high", "explanation": "...", "is_ambiguous": false, "clarification_question": null}}"""

    def evaluate_arguments(
        self,
        risk_input: RiskInput,
        arguments: List[Argument],
        clarification_answer: Optional[str] = None,
    ) -> JudgeDecision:
        self.log("Evaluating arguments...")
        clarification_context = ""
        if clarification_answer:
            clarification_context = f"\n\nADDITIONAL INFO:\n{clarification_answer}"
        arguments_text = "\n\n".join(
            f"=== {arg.agent_name} (for {arg.risk_level.upper()}) ===\nReasoning: {arg.reasoning}\nEvidence: {json.dumps(arg.evidence)}\nConfidence: {arg.confidence:.0%}\nAcknowledges weaknesses: {arg.acknowledges_weaknesses}"
            for arg in arguments
        )
        user_prompt = f"""PROJECT/LOAN:\n{json.dumps(risk_input.to_dict(), indent=2)}\n\nARGUMENTS:\n{arguments_text}{clarification_context}\n\nScore each argument and determine the winner. JSON only."""

        response = self.client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": self.get_judge_prompt()},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
            response_format={"type": "json_object"},
        )
        result = json.loads(response.choices[0].message.content)
        scores = [
            JudgeScore(
                agent_name=s["agent_name"],
                evidence_grounding=s["evidence_grounding"],
                internal_consistency=s["internal_consistency"],
                domain_plausibility=s["domain_plausibility"],
                counterpoint_handling=s["counterpoint_handling"],
                confidence_calibration=s["confidence_calibration"],
                total_score=s["total_score"],
            )
            for s in result["scores"]
        ]
        decision = JudgeDecision(
            final_label=result["final_label"].lower(),
            explanation=result["explanation"],
            scores=scores,
            is_ambiguous=result.get("is_ambiguous", False),
            clarification_question=result.get("clarification_question"),
        )
        self.log(f"VERDICT: {decision.final_label.upper()}")
        return decision

In [ ]:
class DebateOrchestrator:
    """Runs single-round debate: advocates -> judge -> decision."""

    def __init__(self, client: OpenAI):
        self.client = client
        self.low_advocate = LowRiskAdvocateAgent(client)
        self.medium_advocate = MediumRiskAdvocateAgent(client)
        self.high_advocate = HighRiskAdvocateAgent(client)
        self.judge = JudgeAgent(client)
        self.debate_log = []

    def run_debate(
        self,
        risk_input: RiskInput,
        clarification_answer: Optional[str] = None,
    ) -> Tuple[JudgeDecision, List[Argument]]:
        self.debate_log = []
        title = risk_input.project_name or risk_input.description[:50] + "..."
        self.add_to_log("SYSTEM", f"Starting debate for: {title}", "white")

        advocates = [self.low_advocate, self.medium_advocate, self.high_advocate]
        arguments = []
        for advocate in advocates:
            arg = advocate.build_argument(risk_input)
            arguments.append(arg)
            self.add_to_log(
                advocate.name,
                f"**Argues {arg.risk_level.upper()} risk** (Confidence: {arg.confidence:.0%})\n\n{arg.reasoning}",
                advocate.color,
            )

        decision = self.judge.evaluate_arguments(risk_input, arguments, clarification_answer)
        scores_text = "\n".join(
            f"  • **{s.agent_name}**: {s.total_score:.2f}/5.0 (EG:{s.evidence_grounding:.2f} IC:{s.internal_consistency:.2f} DP:{s.domain_plausibility:.2f} CH:{s.counterpoint_handling:.2f} CC:{s.confidence_calibration:.2f})"
            for s in sorted(decision.scores, key=lambda x: x.total_score, reverse=True)
        )
        judge_message = f"**VERDICT: {decision.final_label.upper()} RISK**\n\n**Scores:**\n{scores_text}\n\n**Reasoning:**\n{decision.explanation}"
        if decision.clarification_question:
            judge_message += f"\n\n**Clarification needed:**\n{decision.clarification_question}"
        self.add_to_log("Judge", judge_message, self.judge.color)
        return decision, arguments

    def add_to_log(self, agent: str, message: str, color: str):
        self.debate_log.append({"agent": agent, "message": message, "color": color})


orchestrator = DebateOrchestrator(openai_client)
print("Debate orchestrator ready")

In [ ]:
sample = risk_cases[0]
risk_input = RiskInput(
    description=sample["description"],
    project_name=sample.get("project_name"),
    sector=sample.get("sector"),
    amount=sample.get("amount"),
)
decision, arguments = orchestrator.run_debate(risk_input)
print(f"\nPredicted: {decision.final_label.upper()} | Actual: {sample['risk_level'].upper()} | Match: {decision.final_label == sample['risk_level']}")

In [ ]:
def evaluate_orchestrator(orc, cases: List[dict], max_n: Optional[int] = None) -> float:
    """Run debate on each case; return accuracy (predicted == actual)."""
    cases = cases[:max_n] if max_n else cases
    correct = 0
    for i, row in enumerate(cases):
        ri = RiskInput(
            description=row["description"],
            project_name=row.get("project_name"),
            sector=row.get("sector"),
            amount=row.get("amount"),
        )
        decision, _ = orc.run_debate(ri)
        if decision.final_label == row["risk_level"]:
            correct += 1
    return correct / len(cases) if cases else 0.0


In [ ]:
COLOR_MAP = {
    Agent.GREEN: "#00ff00",
    Agent.YELLOW: "#ffff00",
    Agent.MAGENTA: "#ff00ff",
    Agent.CYAN: "#00ffff",
    "white": "#ffffff",
}


def format_debate_log(debate_log):
    html_parts = []
    for entry in debate_log:
        color = COLOR_MAP.get(entry["color"], "#ffffff")
        agent = entry["agent"]
        message = entry["message"].replace("\n", "<br>")
        html_parts.append(
            f'<div style="margin:15px 0;padding:15px;background:#1a1a1a;border-left:4px solid {color};border-radius:5px;">'
            f'<div style="color:{color};font-weight:bold;font-size:16px;margin-bottom:10px;">[{agent}]</div>'
            f'<div style="color:#e0e0e0;line-height:1.6;">{message}</div></div>'
        )
    return "<div style='background:#0a0a0a;padding:20px;border-radius:10px;'>" + "".join(html_parts) + "</div>"


def run_risk_debate(description, project_name, sector, amount_str, clarification):
    if not (description or "").strip():
        return "Please enter a project/loan description.", "—"
    amount = None
    if amount_str and str(amount_str).strip():
        try:
            amount = float(amount_str)
        except ValueError:
            pass
    risk_input = RiskInput(
        description=description.strip(),
        project_name=project_name.strip() or None,
        sector=sector.strip() or None,
        amount=amount,
    )
    clarification_answer = clarification.strip() or None
    decision, arguments = orchestrator.run_debate(risk_input, clarification_answer)
    debate_html = format_debate_log(orchestrator.debate_log)
    summary = f"## Final classification: **{decision.final_label.upper()} RISK**\n\n### Scores\n"
    for s in sorted(decision.scores, key=lambda x: x.total_score, reverse=True):
        summary += f"- **{s.agent_name}**: {s.total_score:.2f}/5.0\n"
    summary += f"\n### Judge's explanation\n{decision.explanation}"
    if decision.clarification_question:
        summary += f"\n\n### Clarification needed\n{decision.clarification_question}"
    return debate_html, summary


def load_random_case():
    row = random.choice(risk_cases)
    def _str(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ""
        return str(v).strip()
    return (
        _str(row.get("description", "")),
        _str(row.get("project_name")),
        _str(row.get("sector")),
        _str(row.get("amount")),
        f"Actual risk: {row['risk_level'].upper()}",
    )

In [ ]:
with gr.Blocks(title="Risk-Level Debate", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Multi-Agent Risk-Level Debate (Low / Medium / High)\n\nThree advocates argue risk level; the Judge decides. **Risk levels:** Low, Medium, High.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## Input")
            description_in = gr.Textbox(label="Project/loan description", placeholder="Paste or type a short description...", lines=5)
            project_name_in = gr.Textbox(label="Project name (optional)", placeholder="e.g. Solar PPA Project")
            sector_in = gr.Textbox(label="Sector (optional)", placeholder="e.g. Energy")
            amount_in = gr.Textbox(label="Amount (optional)", placeholder="e.g. 500000")
            clarification_in = gr.Textbox(label="Clarification answer (if Judge asked)", placeholder="Optional", lines=2)
            with gr.Row():
                run_btn = gr.Button("Run debate", variant="primary")
                load_btn = gr.Button("Load random case")
            actual_label_out = gr.Textbox(label="Actual label (when using Load random)", interactive=False)
        with gr.Column(scale=2):
            gr.Markdown("## Debate transcript")
            debate_out = gr.HTML()
    gr.Markdown("## Decision summary")
    summary_out = gr.Markdown()

    run_btn.click(
        fn=run_risk_debate,
        inputs=[description_in, project_name_in, sector_in, amount_in, clarification_in],
        outputs=[debate_out, summary_out],
    )
    load_btn.click(
        fn=load_random_case,
        inputs=[],
        outputs=[description_in, project_name_in, sector_in, amount_in, actual_label_out],
    )

demo.launch(inbrowser=True, share=False)